# Project OmniMarket: Industrial Ingestion Architecture
### Engineering Contract:
* Zero external library dependencies (Pure standard library).
* Complete decoupling of Network, Transformation, and Disk I/O layers.
* Explicit fault isolation via type-hinted functional gates.

In [32]:
import urllib.request
import urllib.error
import json
import sys
import time

# =====================================================================
# MODULE 1: THE NETWORK INGESTION ENGINE (Owned by Engineer A)
# =====================================================================
def fetch_raw_network_payload(target_url: str) -> str:
    """
    Establishes raw socket connections with handling for HTTP errors and timeouts.
    RETURNS: Raw UTF-8 string data, or None if the network layer fails.
    """
    max_retries = 3
    initial_delay = 2
    
    # Enforce standard browser headers so firewalls do not instantly block our sandbox
    request_headers = {'User-Agent': 'Mozilla/5.0 (OmniMarket.Engine/2026)'}
    req = urllib.request.Request(target_url, headers=request_headers)
    
    for attempt in range(1, max_retries + 1):
        try:
            print(f"NETWORK: Attempting connection {attempt}/{max_retries}...")
            with urllib.request.urlopen(req, timeout=10) as response:
                if response.status == 200:
                    print("NETWORK: Stream secured. Pulling bytes to memory.")
                    return response.read().decode('utf-8')
                    
        except urllib.error.HTTPError as http_err:
            print(f"NETWORK WARNING: Server rejected request with code {http_err.code} ({http_err.reason})", file=sys.stderr)
            if http_err.code in [429, 503] and attempt < max_retries:
                wait_time = initial_delay * (2 ** (attempt - 1))
                print(f"NETWORK: Rate limit triggered. Backing off for {wait_time}s...", file=sys.stderr)
                time.sleep(wait_time)
            else:
                break # Fatal HTTP status or retries exhausted, abort loop
                
        except Exception as general_err:
            print(f"NETWORK WARNING: Physical connection drop: {general_err}", file=sys.stderr)
            if attempt < max_retries:
                time.sleep(initial_delay)
                
    print("NETWORK CRITICAL: Ingestion pipeline failure. Returning empty buffer.", file=sys.stderr)
    return None

# =====================================================================
# MODULE 2: STRUCTURAL INTEGRATION LOGIC (Owned by Engineer B)
# =====================================================================
import json
import sys

# =====================================================================
# UPDATED MODULE 2: RE-ENGINEERED DEFENSIVE SANITIZATION (Agent Approved)
# =====================================================================
def sanitize_raw_string_strict(raw_string: str, parent_key: str, dataset_type: str) -> list:
    """
    Validates schemas dynamically based on the data origin.
    Enforces a strict contract: Rejects any row missing an ID or Origin source.
    """
    if not raw_string:
        return []
        
    try:
        compiled_payload = json.loads(raw_string)
    except json.JSONDecodeError as parse_error:
        print(f"QUALITY CRITICAL: Structural layout corruption: {parse_error}", file=sys.stderr)
        return []
        
    raw_records = compiled_payload.get(parent_key, compiled_payload) if isinstance(compiled_payload, dict) else compiled_payload
    
    sanitized_batch = []
    quarantine_counter = 0
    
    for item in raw_records:
        # DYNAMIC SCHEMA MAPPING CONTRATS:
        if dataset_type == "spaceflight":
            entity_id = item.get("id")
            title = item.get("title")
            origin = item.get("news_site")
        elif dataset_type == "food":
            entity_id = item.get("code")
            title = item.get("product_name")
            origin = item.get("brands")
        else:
            entity_id = item.get("id")
            title = item.get("title")
            origin = item.get("source_origin")

        # AGENT DEPLOYMENT GUARD 5: STRICT CRITICAL FIELD VALIDATION
        # If the ID, Title, or Origin is completely missing or blank, QUARANTINE IT!
        if not entity_id or not title or not origin:
            quarantine_counter += 1
            # Log the dropped line metadata defensively to the system console
            print(f"QA ALERT: Row dropped. Failed strict data validation contract. Identifier Null.", file=sys.stderr)
            continue
            
        clean_record = {
            "entity_id": str(entity_id).strip(),
            "product_name": str(title).strip(),
            "source_origin": str(origin).strip()
        }
        sanitized_batch.append(clean_record)
        
    print(f"\n--- DATA QUALITY INGESTION REPORT ---")
    print(f"Successfully Sanitized Rows Committed : {len(sanitized_batch)}")
    print(f"Corrupt/Incomplete Rows Quarantined     : {quarantine_counter}")
    print(f"------------------------------------\n")
    return sanitized_batch
#=====================================================================
# MODULE 3: PERSISTENCE VAULT OPERATOR (Owned by Engineer A)
# =====================================================================
def commit_batch_to_disk(clean_batch: list, output_filepath: str):
    """
    Commits sanitized records directly to physical disk storage with instant flushing.
    """
    if not clean_batch:
        print("STORAGE WARNING: Operation bypassed. Received empty payload batch.", file=sys.stderr)
        return
        
    try:
        print(f"STORAGE: Opening connection handle to '{output_filepath}'...")
        with open(output_filepath, mode='a', encoding='utf-8') as vault_file:
            for record in clean_batch:
                flat_text_line = json.dumps(record)
                vault_file.write(flat_text_line + '\n')
            
            # Immediately force flush the OS buffer from RAM to disk
            vault_file.flush()
        print(f"🚀 STORAGE SUCCESS: Committed {len(clean_batch)} records to physical storage.")
        
    except IOError as disk_error:
        print(f"STORAGE CRITICAL: Write operation rejected by hardware: {disk_error}", file=sys.stderr)

# =====================================================================
# THE PIPELINE ORCHESTRATION HUB
# =====================================================================
if __name__ == "__main__":
    # Test target: Global spaceflight news database (unthrottled, stable testing source)
    TARGET_ENDPOINT = "https://api.spaceflightnewsapi.net/v4/articles/?limit=10"
    TARGET_KEY = "results"
    VAULT_FILE = "market_vault.jsonl"
    
    print("--- STEP 1: HARVESTING LIVE NETWORK STREAM ---")
    raw_buffer = fetch_raw_network_payload(TARGET_ENDPOINT)
    
    print("\n--- STEP 2: PARSING & SANITIZING BUFFER SCHEMA ---")
    sanitized_data = sanitize_raw_string(raw_buffer, parent_key=TARGET_KEY)
    
    print("\n--- STEP 3: COMMITTING UNTO STORAGE PERMANENCE ---")
    commit_batch_to_disk(sanitized_data, VAULT_FILE)

--- STEP 1: HARVESTING LIVE NETWORK STREAM ---
NETWORK: Attempting connection 1/3...
NETWORK: Stream secured. Pulling bytes to memory.

--- STEP 2: PARSING & SANITIZING BUFFER SCHEMA ---
QUALITY: Data schema normalized. Isolated 10 objects.

--- STEP 3: COMMITTING UNTO STORAGE PERMANENCE ---
STORAGE: Opening connection handle to 'market_vault.jsonl'...
🚀 STORAGE SUCCESS: Committed 10 records to physical storage.


In [33]:
# =====================================================================
# MODULE 4: THE STREAMING ANALYTICS CORE (Owned by Engineer B)
# =====================================================================
import json
import sys

def compute_warehouse_metrics(input_filepath: str):
    """
    Streams local data from disk line-by-line and calculates structural frequency aggregations.
    Uses zero external libraries and maintains an O(1) memory footprint profile.
    """
    print(f"ANALYTICS: Opening data warehouse stream from '{input_filepath}'...")
    
    total_record_count = 0
    source_frequency_table = {} # In-memory dictionary acting as our accumulator table
    
    try:
        # 'r' mode stands for READ-ONLY. Protects our archive file from accidental writes.
        with open(input_filepath, mode='r', encoding='utf-8') as vault_read:
            
            for line_number, raw_line in enumerate(vault_read, start=1):
                clean_line_text = raw_line.strip()
                
                # Skip blank separator padding lines if they exist
                if not clean_line_text:
                    continue
                    
                try:
                    # Deserialize the text line back into a dynamic dictionary
                    record_dict = json.loads(clean_line_text)
                except json.JSONDecodeError:
                    print(f"ANALYTICS WARNING: Skipping corrupted row discovered at line {line_number}", file=sys.stderr)
                    continue
                
                # Increment global tracker metrics
                total_record_count += 1
                
                # Extract the tracking origin field we built in Module 2
                origin = record_dict.get("source_origin", "UNKNOWN_SOURCE")
                
                # ACCUMULATION PROTOCOL: If key is new, initialize it at 1. Else, increment it.
                if origin not in source_frequency_table:
                    source_frequency_table[origin] = 1
                else:
                    source_frequency_table[origin] += 1
                    
        # --- TELEMETRY DASHBOARD DISPLAY ---
        print("\n" + "="*40)
        print("          OMNIMARKET BI DASHBOARD      ")
        print("="*40)
        print(f"Total Database Rows Ingested : {total_record_count}")
        print("-"*40)
        print("Distribution Breakdown by Origin/Brand:")
        
        for origin_name, count in source_frequency_table.items():
            # Formatting values cleanly for clear display layout
            print(f" -> {origin_name[:25]:<25} : {count} records")
        print("="*40 + "\n")
        
    except FileNotFoundError:
        print(f"ANALYTICS CRITICAL: Target file '{input_filepath}' does not exist on disk.", file=sys.stderr)

# =====================================================================
# METRICS RUNNER
# =====================================================================
if __name__ == "__main__":
    WAREHOUSE_FILE = "market_vault.jsonl"
    compute_warehouse_metrics(WAREHOUSE_FILE)

ANALYTICS: Opening data warehouse stream from 'market_vault.jsonl'...

          OMNIMARKET BI DASHBOARD      
Total Database Rows Ingested : 100
----------------------------------------
Distribution Breakdown by Origin/Brand:
 -> UNKNOWN_SOURCE            : 100 records



In [34]:
# =====================================================================
# MODULE 5: ATOMIC CSV CONVERSION ENGINE (Owned by Engineer A)
# =====================================================================
import json
import sys

def convert_vault_to_csv(input_jsonl_path: str, output_csv_path: str):
    """
    Streams structured rows from JSON Lines format and flushes them 
    into a clean CSV document layout with defensive string escape controls.
    """
    print(f"CONVERTER: Transforming '{input_jsonl_path}' into '{output_csv_path}'...")
    
    # Define our strict column tracking order (The Schema Contract)
    headers = ["entity_id", "product_name", "source_origin"]
    
    try:
        with open(input_jsonl_path, mode='r', encoding='utf-8') as jsonl_source, \
             open(output_csv_path, mode='w', encoding='utf-8') as csv_target:
                 
            # 1. Commit the tracking header row to the top of the file
            header_row = ",".join(headers) + "\n"
            csv_target.write(header_row)
            
            row_counter = 0
            for line in jsonl_source:
                clean_line = line.strip()
                if not clean_line:
                    continue
                    
                try:
                    record = json.loads(clean_line)
                except json.JSONDecodeError:
                    continue # Skip broken lines defensively
                
                # 2. Extract values based strictly on our header order contract
                row_values = []
                for field in headers:
                    val = str(record.get(field, "")).strip()
                    
                    # Escape Rule: If a text field contains a comma, wrap it in double quotes 
                    # so Excel doesn't accidentally split it into two columns!
                    if "," in val or '"' in val:
                        val = val.replace('"', '""') # Double up internal quotes
                        val = f'"{val}"'
                        
                    row_values.append(val)
                
                # 3. Assemble and commit the CSV line row
                csv_line = ",".join(row_values) + "\n"
                csv_target.write(csv_line)
                row_counter += 1
                
            csv_target.flush() # Force write to physical storage platter
            print(f"🚀 CONVERTER SUCCESS: Exported {row_counter} data rows to '{output_csv_path}'.")
            
    except FileNotFoundError:
        print(f"CONVERTER CRITICAL: Source asset '{input_jsonl_path}' not found.", file=sys.stderr)

# =====================================================================
# EXECUTE CSV EXPORT
# =====================================================================
if __name__ == "__main__":
    convert_vault_to_csv("market_vault.jsonl", "market_analytics.csv")

CONVERTER: Transforming 'market_vault.jsonl' into 'market_analytics.csv'...
🚀 CONVERTER SUCCESS: Exported 100 data rows to 'market_analytics.csv'.
